# `OfnerEEGNet` 학습 코드 (`checkpoint.pt`)

- 데이터셋: moabb `Ofner2017` (Graz upper-limb motor imagery), **subject 1**
- 세션: **imagined-movement 세션만** 사용 (Ofner2017은 기본으로 execution + imagination 두 세션을 모두 반환하므로 imagination만 걸러냄)
- 분할: 세션당 10회 run(`"0"`~`"9"`) 중 **run 8, 9를 validation으로 홀드아웃**, 나머지 8개 run으로 학습
- 전처리: `n2o.decoder.bandpass_standardize()` — EEG 채널만 선택, V→uV, 4–38Hz 대역통과, exponential moving standardize
- 윈도잉: `n2o.decoder.window_by_event()` — trial마다 큐(cue) 시점부터 1개의 이벤트 정렬 윈도우 (`start_offset_sec=0.0, stop_offset_sec=0.0`, 3.0s @ 512Hz = 1536 samples)
- 모델: `n2o.decoder.BraindecodeDecoder("EEGNet", n_chans=61, n_outputs=7, n_times=1536)`


In [ ]:
import warnings

import mne
import numpy as np
import torch
from torch.utils.data import DataLoader

from n2o.decoder import (
    BraindecodeDecoder,
    bandpass_standardize,
    label_names,
    window_by_event,
)
from n2o.signal.dataset import DatasetLoader

warnings.filterwarnings("ignore")  # mne/braindecode의 사소한 UserWarning 숨김
mne.set_log_level("ERROR")

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)

## 1. Subject 1 raw 레코딩 불러오기

`DatasetLoader(name="Ofner2017").read()`는 subject 1의 raw 레코딩만 돌려줍니다 (전처리/윈도잉은 하지 않음 — `signal.read()`는 순수 로더). moabb의 `Ofner2017`은 기본값이 `imagined=True, executed=True`라 execution 세션(`"0execution"`)과 imagination 세션(`"1imagination"`)이 합쳐진 20개 run(BaseConcatDataset)이 돌아옵니다 — `checkpoint.pt`는 imagination 세션만으로 학습했으므로 먼저 세션으로 나눠서 걸러냅니다.


In [ ]:
raw_dataset = DatasetLoader(name="Ofner2017").read()

by_session = raw_dataset.split("session")
print("세션:", list(by_session.keys()))

imagined = by_session["1imagination"]
print("imagination 세션의 run 수:", len(imagined.datasets))

## 2. 전처리 — `bandpass_standardize()`

EEG 채널만 선택하고, V→uV로 바꾼 뒤, 4–38Hz 대역통과 필터와 exponential moving standardize를 적용합니다 (in-place). `OfnerEEGNet`은 `preprocessing_kwargs`를 따로 넘기지 않으므로 기본값(`l_freq=4.0, h_freq=38.0`)을 그대로 씁니다.


In [ ]:
bandpass_standardize(imagined)

## 3. Run 단위로 train/validation 분할

`_get_single_subject_data()`가 run마다 문자열 인덱스(`"0"`~`"9"`, 10개 run)를 매기므로 `.split("run")`로 그대로 나눌 수 있습니다. `checkpoint.pt`는 run 8, 9를 validation으로 홀드아웃하고 나머지 8개 run(`"0"`~`"7"`)으로 학습했습니다.


In [ ]:
by_run = imagined.split("run")
print("run:", sorted(by_run.keys(), key=int))

VAL_RUNS = {"8", "9"}
train_runs = [ds for run, ds in by_run.items() if run not in VAL_RUNS]
val_runs = [ds for run, ds in by_run.items() if run in VAL_RUNS]

from itertools import chain

from braindecode.datasets import BaseConcatDataset

train_raw = BaseConcatDataset(
    list(chain.from_iterable(ds.datasets for ds in train_runs))
)
val_raw = BaseConcatDataset(list(chain.from_iterable(ds.datasets for ds in val_runs)))
print("train run 수:", len(train_raw.datasets), " / val run 수:", len(val_raw.datasets))

## 4. 윈도잉 — `window_by_event()`

trial(cue)마다 이벤트 정렬 윈도우 1개씩 잘라냅니다. `OfnerEEGNet`의 `windowing_kwargs`대로 `start_offset_sec=0.0, stop_offset_sec=0.0` — Ofner2017 자체의 cue-relative interval이 `[0, 3]`이므로 트라이얼 전체(3.0s = 512Hz 기준 1536 samples)가 그대로 한 윈도우가 됩니다. `label_names()`로 클래스 인덱스 순서대로 정렬된 라벨 목록을 읽어옵니다 — `Classification.window()`가 내부적으로 하는 것과 동일합니다.


In [ ]:
WINDOWING_KWARGS = {"start_offset_sec": 0.0, "stop_offset_sec": 0.0}

train_windows = window_by_event(train_raw, **WINDOWING_KWARGS)
val_windows = window_by_event(val_raw, **WINDOWING_KWARGS)

labels = label_names(train_windows)
print("labels:", labels)
print("train windows:", len(train_windows), " / val windows:", len(val_windows))

x0, y0, _ = train_windows[0]
print("윈도우 shape:", x0.shape)  # (n_chans, n_times) = (61, 1536) 이어야 함

## 5. 모델 — `BraindecodeDecoder("EEGNet", ...)`

학습되지 않은 `EEGNet`을 지정된 shape(`n_chans=61, n_outputs=7, n_times=1536`)로 만듭니다. `decoder.model`이 실제 `torch.nn.Module`이므로 이걸 직접 학습시킵니다.


In [ ]:
decoder = BraindecodeDecoder(
    "EEGNet", n_chans=61, n_outputs=7, n_times=1536, windowing_kwargs=WINDOWING_KWARGS
)
model = decoder.model
print(
    model.__class__.__name__,
    "파라미터 개수:",
    sum(p.numel() for p in model.parameters()),
)

## 6. 학습 루프

평범한 PyTorch 학습 루프입니다. Adam + cross-entropy, 매 epoch마다 held-out run(8, 9)에 대한 validation accuracy를 확인합니다.


In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

train_loader = DataLoader(train_windows, batch_size=16, shuffle=True)
val_loader = DataLoader(val_windows, batch_size=16, shuffle=False)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
criterion = torch.nn.CrossEntropyLoss()
N_EPOCHS = 30

In [ ]:
def run_epoch(loader, train):
    model.train(train)
    total, correct, loss_sum = 0, 0, 0.0
    for X, y, _ in loader:
        X = X.float().to(device)
        y = y.long().to(device)
        with torch.set_grad_enabled(train):
            logits = model(X)
            loss = criterion(logits, y)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        loss_sum += loss.item() * y.size(0)
        correct += (logits.argmax(dim=1) == y).sum().item()
        total += y.size(0)
    return loss_sum / total, correct / total


for epoch in range(1, N_EPOCHS + 1):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    print(
        f"epoch {epoch:2d}  train loss {train_loss:.4f} acc {train_acc:.3f}  |  "
        f"val loss {val_loss:.4f} acc {val_acc:.3f}"
    )

## 7. 체크포인트 저장

`OfnerEEGNet.__init__()`이 그대로 읽어들일 수 있는 형태 — `model.state_dict()`를 plain `torch.save()`로 저장합니다. `decoder.config.labels`도 `OfnerEEGNet`이 하드코딩한 라벨 순서와 일치하는지 확인합니다.


In [ ]:
decoder.config.labels = labels
assert labels == [
    "rest",
    "right_elbow_extension",
    "right_elbow_flexion",
    "right_hand_close",
    "right_hand_open",
    "right_pronation",
    "right_supination",
], (
    "라벨 순서가 OfnerEEGNet의 하드코딩된 _LABELS와 다릅니다 -- ofner_eegnet.py도 함께 갱신하세요"
)

model.to("cpu")
torch.save(model.state_dict(), "checkpoint.pt")
print("checkpoint.pt 저장 완료")

## 8. 위치 변경

`src/n2o/decoder/classification/ofner_eegnet/checkpoint.pt`로 옮긴 뒤 `OfnerEEGNet()`을 생성하면 됩니다.
